In [1]:
import requests
import pandas as pd

# ==========================================
# 1. PARÂMETROS DA BUSCA
# ==========================================
lat = -22.90
lon = -47.06
data_inicio = '20230801' # Formato da NASA é YYYYMMDD
data_fim = '20230831'

# WS50M = Velocidade do vento a 50m / WD50M = Direção do vento a 50m
parametros = 'WS50M,WD50M'

# Montando a URL da API
url = f"https://power.larc.nasa.gov/api/temporal/daily/point?parameters={parametros}&community=RE&longitude={lon}&latitude={lat}&start={data_inicio}&end={data_fim}&format=JSON"

print("⏳ Buscando dados na NASA...")

# ==========================================
# 2. REQUISIÇÃO E TRATAMENTO
# ==========================================
resposta = requests.get(url)
dados_json = resposta.json()

# O JSON da NASA vem um pouco aninhado, extraímos direto os dicionários de vento
dict_velocidade = dados_json['properties']['parameter']['WS50M']
dict_direcao = dados_json['properties']['parameter']['WD50M']

# ==========================================
# 3. CONVERSÃO PARA PANDAS E LIMPEZA
# ==========================================
# Criamos a tabela cruzando as duas informações
df_vento = pd.DataFrame({
    'Velocidade_Vento_50m': dict_velocidade,
    'Direcao_Vento_50m': dict_direcao
})

# A data vem como o índice da tabela (ex: '20230801'), transformamos em coluna
df_vento = df_vento.reset_index()
df_vento = df_vento.rename(columns={'index': 'Data_Bruta'})

# Formatando a data para ficar idêntica à do satélite (YYYY-MM-DD)
df_vento['Data'] = pd.to_datetime(df_vento['Data_Bruta'], format='%Y%m%d').dt.strftime('%Y-%m-%d')
df_vento = df_vento.drop(columns=['Data_Bruta'])

# Reordenando as colunas para ficar visualmente limpo
df_vento = df_vento[['Data', 'Velocidade_Vento_50m', 'Direcao_Vento_50m']]

# ==========================================
# 4. SALVAR RESULTADO
# ==========================================
df_vento.to_csv('historico_vento_campinas.csv', index=False)

print("✅ Dados de vento extraídos com sucesso! Arquivo CSV gerado.")
display(df_vento.head())

⏳ Buscando dados na NASA...
✅ Dados de vento extraídos com sucesso! Arquivo CSV gerado.


,Data,Velocidade_Vento_50m,Direcao_Vento_50m
0,2023-08-01,4.42,110.7
1,2023-08-02,4.05,8.5
2,2023-08-03,4.69,38.2
3,2023-08-04,4.40,137.7
4,2023-08-05,3.45,119.5
